In [36]:
import sys
sys.path.append('..')

In [37]:
import pandas as pd
import pickle
from sklearn.model_selection import TimeSeriesSplit
from wrappers.random_forrest_forecaster_wrapper import RandomForrestForecaster
from sklearn.metrics import mean_absolute_percentage_error
from collections import defaultdict

In [38]:
with open('../../data/datasets/data_cleaned.pkl', 'rb') as f:
    states_dfs = pickle.load(f)

Two good sarima states and three bad sarima states

In [39]:
states = ['NV','LA', 'HI', 'MO', 'IA']
horizon= 24
target = 'residential_electricity_price'

In [40]:
kfold = TimeSeriesSplit(n_splits=3, test_size=24)

In [41]:
components = [2, 3, 5, 7]
cross_val_metrics = {}
for state in states:
    # hold out the last 24 months
    cross_val_metrics[state] = defaultdict(list)
    df = states_dfs[state].iloc[:-horizon]
    for comp in components:
        for i, (train_index, test_index) in enumerate(kfold.split(df)):
            X_tt = df.iloc[train_index].drop(columns=[target])
            y_tt  = df.iloc[train_index][target]
            y_val = df.iloc[test_index][target]
            model = RandomForrestForecaster(n_componets=comp, horizon=horizon)
            model.fit(X_tt, y_tt)
            preds = model.predict()
            mape = mean_absolute_percentage_error(y_val, preds)
            cross_val_metrics[state][comp].append(mape)

100%|██████████| 7/7 [36:15<00:00, 310.83s/it]


In [42]:
with open('rf_cross_val.pkl', 'wb') as f:
    pickle.dump(cross_val_metrics, f)

cross_val_metrics

{'NV': defaultdict(list,
             {2: [0.046564876630219874,
               0.046091800547217565,
               0.19046910434537268],
              3: [0.0400797172469359,
               0.04349775840492476,
               0.19443495069417394],
              5: [0.04707626418919932,
               0.03834156526060216,
               0.18675445674051475],
              7: [0.03499858008561029,
               0.06162859175116118,
               0.170046063210344]}),
 'LA': defaultdict(list,
             {2: [0.036328686427926436,
               0.047390402088316186,
               0.10009096536363993],
              3: [0.036786998595223254,
               0.04790122525602943,
               0.1245912863492087],
              5: [0.03655920908286412,
               0.04756488914071486,
               0.15001903710409567],
              7: [0.03528485243913964,
               0.052316978767619114,
               0.15312962462205304]}),
 'HI': defaultdict(list,
             {2: [0.072

In [45]:
import numpy as np

In [46]:
for state in states:
    for comp in components:
        print(f'state: {state}, components: {comp}, average mape: {np.mean(cross_val_metrics[state][comp])}')

state: NV, components: 2, average mape: 0.09437526050760336
state: NV, components: 3, average mape: 0.09267080878201155
state: NV, components: 5, average mape: 0.09072409539677206
state: NV, components: 7, average mape: 0.08889107834903849
state: LA, components: 2, average mape: 0.06127001795996085
state: LA, components: 3, average mape: 0.06975983673348712
state: LA, components: 5, average mape: 0.07804771177589155
state: LA, components: 7, average mape: 0.08024381860960393
state: HI, components: 2, average mape: 0.1262031664055094
state: HI, components: 3, average mape: 0.12530664753612822
state: HI, components: 5, average mape: 0.12931178349456354
state: HI, components: 7, average mape: 0.12815285803527432
state: MO, components: 2, average mape: 0.13035253093047003
state: MO, components: 3, average mape: 0.12360824652799933
state: MO, components: 5, average mape: 0.07636341327413354
state: MO, components: 7, average mape: 0.09597643606704859
state: IA, components: 2, average mape: 0